# Phase 6 - Prediction and submission
Load the final model (Phase 5), predict the 5,014 test deliveries, check the result against the template contract and against validation behaviour, and write `submission_task1.csv`.

**Template contract (booklet):** keep `delivery_id` values and row order exactly as supplied, no rows added or removed, fill only `pred_service_min` (minutes) and `pred_late_prob` (0 to 1).

In [ ]:
import sys, json; sys.path.append("../../tools/task1")
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from task1_common import *
import task1_models as tm
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
t = load_task1_tables()

## 1. Load the model and the test features

In [ ]:
models = joblib.load(MODELS / 'task1_models.joblib')
test_X = pd.read_pickle(PROCESSED / 'task1_test_features.pkl')
meta = json.load(open(PROCESSED / 'task1_feature_meta.json'))
template = t['template']
assert models['features'] == meta['features'], 'model and feature table were built with different feature lists'
assert set(meta['features']) <= set(test_X.columns)
print('models:', [k for k in models if not k.startswith('f') and k != 'segment_brands'])
print('test features:', test_X.shape, '| template rows:', len(template))
print('test ids == template ids (as a set):', set(test_X.delivery_id) == set(template.delivery_id))

## 2. Predict

In [ ]:
svc, late, parts = tm.predict(models, test_X, t['traffic'], return_parts=True)
pred = pd.DataFrame({'delivery_id': test_X.delivery_id.values, 'pred_service_min': svc, 'pred_late_prob': late})
print(pred.describe().round(3))

## 3. Build the submission in template order

In [ ]:
sub = template[['delivery_id']].merge(pred, on='delivery_id', how='left', validate='one_to_one')
sub['pred_service_min'] = sub.pred_service_min.round(2)
sub['pred_late_prob'] = sub.pred_late_prob.clip(0, 1).round(4)
validate_submission(sub, template)          # columns, row count, id order, no NaN, ranges
print('template contract: OK')
sub.head(10)

## 4. Sanity checks against validation behaviour
The test window is monsoon-heavy (66%) with more road disruption, so its expected late rate is ~21-23% (Phase 2/3), above the 19.6% training average.

In [ ]:
chk = test_X[['delivery_id', 'brand', 'district', 'monsoon', 'dock_type', 'service_allowance_min', 'plan_slack', 'sim_hist_slack']].merge(sub, on='delivery_id')
for c in ['brand', 'district', 'dock_type']:
    chk[c] = chk[c].astype(str)
train_lab = pd.read_pickle(INTERIM / 'train_labeled.pkl')
fm = train_lab[pd.to_datetime(train_lab.date).dt.month.isin([2, 3])]

print('mean predicted late prob (test):', round(chk.pred_late_prob.mean(), 4))
print('reference - train Feb-Mar late rate by monsoon:', fm.groupby('monsoon').late.mean().round(4).to_dict(),
      '-> weighted to test mix:', round((fm.groupby('monsoon').late.mean() * chk.monsoon.value_counts(normalize=True)).sum(), 4))
print('\nby brand: predicted vs training history')
by_brand = pd.DataFrame({'test_pred_service': chk.groupby('brand').pred_service_min.mean(), 'train_service': train_lab.groupby('brand').service_min.mean(),
                         'test_pred_late': chk.groupby('brand').pred_late_prob.mean(), 'train_late': train_lab.groupby('brand').late.mean(),
                         'n_test': chk.brand.value_counts()})
print(by_brand.round(3))
print('\nlate prob by monsoon (test):', chk.groupby('monsoon').pred_late_prob.mean().round(4).to_dict())
print('service vs dispatcher allowance (test mean):', round(chk.pred_service_min.mean(), 2), 'vs', round(chk.service_allowance_min.mean(), 2))

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
chk.pred_late_prob.hist(bins=50, ax=ax[0]); ax[0].set_title('pred_late_prob distribution (test)')
for b, g in chk.groupby('brand'): g.pred_service_min.plot.hist(bins=50, alpha=.5, density=True, label=b, ax=ax[1])
ax[1].legend(); ax[1].set_title('pred_service_min by brand (test)')
bins = pd.cut(chk.sim_hist_slack, np.arange(-120, 241, 20)); r = chk.groupby(bins, observed=True).pred_late_prob.mean()
ax[2].plot([i.mid for i in r.index], r.values, marker='o'); ax[2].set_title('pred late prob vs simulated slack (should fall)'); ax[2].set_xlabel('slack (min)')
ax[3].scatter(chk.service_allowance_min, chk.pred_service_min, s=4, alpha=.3); ax[3].plot([0, 80], [0, 80], 'r--')
ax[3].set_title('pred service vs dispatcher allowance'); ax[3].set_xlabel('allowance (min)'); ax[3].set_ylabel('predicted (min)')
plt.tight_layout(); plt.show()
corr = np.corrcoef(chk.sim_hist_slack, chk.pred_late_prob)[0, 1]
print(f'corr(simulated slack, late prob) = {corr:.3f}  (expected strongly negative)')

## 5. Write the submission
The booklet's example shows `ORD0092308` first - the same id that heads our template.

In [ ]:
PREDICTIONS.mkdir(parents=True, exist_ok=True)
out = PREDICTIONS / 'submission_task1.csv'
sub.to_csv(out, index=False)

back = pd.read_csv(out)                     # re-read from disk and re-validate exactly what will be submitted
validate_submission(back, template)
print('written and re-validated:', out, back.shape)
print(back.head(3).to_string(index=False))

### Submission notes
| Check | Result |
|---|---|
| Columns / row count / id order identical to the template | validated twice (in memory and after re-reading the file) |
| `pred_late_prob` in [0, 1], `pred_service_min` > 0, no NaN | validated |
| Predicted late rate consistent with the monsoon-heavy test window | section 4 |
| Late probability falls as simulated slack grows | section 4 |
